# Sampling

## Setup

In [4]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import rioxarray as rxr
import mgrs

In [5]:
import gc

In [6]:
DATA_DIR = '../data'

## Load data tiles

In [7]:
def tileId_to_epsg(tileId):
    zone = int(tileId[:2])
    return f'EPSG:326{zone}'

In [8]:
data_tiles_all = {}
data_tiles_dir = os.path.join(DATA_DIR, 'data_tiles')
for f in os.listdir(data_tiles_dir):
    if not f.startswith('data_tile'):
        continue

    tile_id = f[10:15]

    f_full = os.path.join(data_tiles_dir, f)
    
    ds = xr.open_dataset(f_full, chunks={})

    if 'PercGravel' not in ds.data_vars:
        ds.close()
        continue
    
    data_tiles_all[tile_id] = f_full

## Sample presence and absence

In [9]:
def sample_by_group(
    df,
    group_by,
    N_per_group,
    seed=None,
    replace=False
):
    rng = np.random.default_rng(seed)

    groups = df.groupby(group_by)

    samples = []
    total_collected = 0

    for name, g in groups:
        n = N_per_group

        # only cap by group size if NOT sampling with replacement
        if not replace:
            n = min(n, len(g))

        if n > 0 and len(g) > 0:
            samples.append(
                g.sample(
                    n=n,
                    replace=replace,
                    random_state=seed
                )
            )
            total_collected += n

    sampled = pd.concat(samples, ignore_index=True)
    return sampled.reset_index(drop=True)


In [10]:
def sample_xarray_by_group(
    ds,
    group_by,
    N_per_group,
    bands=None,
    seed=None,
    mask_da=None,
    replace=False,
    N_max=None
):
    rng = np.random.default_rng(seed)

    if bands is None:
        bands = list(ds.data_vars)

    # Lazy valid-data mask
    mask = xr.ones_like(ds[bands[0]], dtype=bool)
    for b in bands:
        mask = mask & ~np.isnan(ds[b])

    if mask_da is not None:
        mask = mask & mask_da

    # Compute mask only
    mask_vals = mask.stack(points=("y", "x")).compute()
    valid_points = np.flatnonzero(mask_vals.values)
    del mask, mask_vals

    if valid_points.size == 0:
        return pd.DataFrame(columns=bands + ["y", "x"])

    y_idx, x_idx = np.unravel_index(valid_points, ds[bands[0]].shape)

    if N_max is not None:
        sample_idx = rng.choice(
            len(y_idx),
            size=N_max,
            replace=False
        )
        y_idx = y_idx[sample_idx]
        x_idx = x_idx[sample_idx]

    # Lazy extraction
    ds_sel = ds.isel(
        y=xr.DataArray(y_idx, dims="points"),
        x=xr.DataArray(x_idx, dims="points")
    )

    df = ds_sel[bands + [group_by]] \
        .to_dataframe().reset_index()

    sampled = sample_by_group(
        df,
        group_by=group_by,
        N_per_group=N_per_group,
        seed=seed,
        replace=replace
    )

    return sampled


In [11]:
import dask

In [12]:
tile_samples = []
N_presence_per_gridtile = 30
N_absence_per_gridtile = 10
threshold_kde = 3.122

for tileId, path in data_tiles_all.items():
    ds = xr.open_dataset(path, chunks={})
    ds = ds.chunk({'y': 2048, 'x': 2048})
    tile_crs = tileId_to_epsg(tile_id)
    ds.rio.write_crs(tile_crs, inplace=True)
    
    if 'ZM_Gauss' not in ds.data_vars:
        ds['ZM_Gauss'] = xr.zeros_like(ds['Rw490'])
    
    print(f"Sampling {tileId}")

    presence_da = xr.where(ds['ZM_Gauss'] >= 0.95, 1, np.nan)  # only sample exact presence points
    absence_da = xr.where(
        (ds['ZM_Gauss'] < 1e-5)&(ds['bathymetry'] > -15.0),  # no upper bathymetry limit to allow for estuaries
        0, np.nan
    )  # minimum distance of 10 pixels from known presence
    da = presence_da.combine_first(absence_da)
    ds['Presence'] = da
    
    presence_samples = sample_xarray_by_group(
        ds,
        group_by='grid_id',
        N_per_group=N_presence_per_gridtile,
        seed=42,
        mask_da=~np.isnan(presence_da),
        replace=True
    )
    presence_samples['tileId'] = tileId
    tile_samples.append(presence_samples)

    absence_samples = sample_xarray_by_group(
        ds,
        group_by='grid_id',
        N_per_group=N_absence_per_gridtile,
        seed=42,
        mask_da=~np.isnan(absence_da),
        replace=False,
        N_max=1000
    )
    absence_samples['tileId'] = tileId
    tile_samples.append(absence_samples)

    # cleanup
    del presence_da, absence_da, da, presence_samples, absence_samples, ds
    gc.collect()

Sampling 29UPB
Sampling 29UPR
Sampling 29VPC
Sampling 29VPE
Sampling 30UUA
Sampling 30UUB
Sampling 30UUC
Sampling 30UUD
Sampling 30UUE
Sampling 30UUF
Sampling 30UUG
Sampling 30UVA
Sampling 30UVB
Sampling 30UVC
Sampling 30UVE
Sampling 30UVF
Sampling 30UWB
Sampling 30UXB
Sampling 30VUH
Sampling 30VUJ
Sampling 30VUK
Sampling 30VVK
Sampling 30VVL
Sampling 30VWL
Sampling 31UCS


In [13]:
combined = pd.concat(tile_samples)

C:\Users\olley\AppData\Local\Temp\ipykernel_19620\1867173721.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined = pd.concat(tile_samples)


In [14]:
len(combined)

10218

In [15]:
combined.head()

,points,latitude,longitude,bitmask,Rnir,Rgli,logchl,logfb,Rw443,Rw490,...,Substrate,PercGravel,PercMud,PercSand,NE_Subtidal,Presence,y,x,spatial_ref,tileId
0,2.0,55.745651,-6.367638,2048.0,0.033210,0.001187,-0.074663,0.539565,0.024355,0.024198,...,4.0,19.901451,16.464964,63.585102,0.0,1.0,6180910.0,665230.0,0.0,29UPB
1,0.0,55.775971,-6.323171,0.0,0.027805,0.001183,0.150560,0.553897,0.019388,0.021580,...,4.0,21.424850,14.333354,64.335358,0.0,1.0,6184390.0,667890.0,0.0,29UPB
2,2.0,55.745651,-6.367638,2048.0,0.033210,0.001187,-0.074663,0.539565,0.024355,0.024198,...,4.0,19.901451,16.464964,63.585102,0.0,1.0,6180910.0,665230.0,0.0,29UPB
3,2.0,55.745651,-6.367638,2048.0,0.033210,0.001187,-0.074663,0.539565,0.024355,0.024198,...,4.0,19.901451,16.464964,63.585102,0.0,1.0,6180910.0,665230.0,0.0,29UPB
4,0.0,55.775971,-6.323171,0.0,0.027805,0.001183,0.150560,0.553897,0.019388,0.021580,...,4.0,21.424850,14.333354,64.335358,0.0,1.0,6184390.0,667890.0,0.0,29UPB


In [16]:
combined['Presence'].value_counts()

Presence
0.0    7788
1.0    2430
Name: count, dtype: int64

In [17]:
combined['tile_grid'] = combined.apply(
    lambda row: row['tileId'] + str(int(row['grid_id'])),
    axis=1
)

In [18]:
combined.to_csv(os.path.join(DATA_DIR, 'samples_v6.csv'))